# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

In [ ]:
# Explore record sets
print("Available record sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '(no name)')}")
    if 'description' in rs:
        print(f"  description: {rs['description']}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - @id: {field.get('@id')}, name: {field.get('name', '(no name)')}")
            else:
                print(f"    - @id: {field}")
    print()

## 3. Data Extraction
Load data from the record sets into DataFrames for analysis. Use the record set and field `@id`s obtained above.

We'll extract *all* record sets automatically below. For this, we'll collect all record set `@id`s into a list and extract the records.

In [ ]:
# Extract all record sets data into DataFrames by @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set @id: {rs_id}")
    else:
        print(f"No records found for record set @id: {rs_id}")

# Display columns of the first loaded DataFrame (if any)
df_keys = list(dataframes.keys())
if df_keys:
    first_rs_id = df_keys[0]
    print(f"Columns in record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common EDA steps: filtering, normalization, grouping by attributes, etc. All field references are by their `@id` as shown in step 2.

> **Note:** For this EDA example, we'll proceed with the first non-empty record set and select the first numeric field available.

In [ ]:
import numpy as np

# Choose the first DataFrame with records
if df_keys:
    eda_rs_id = first_rs_id
    df = dataframes[eda_rs_id]
    # Detect numeric fields (columns where dtype is float or int)
    numeric_fields = [c for c in df.columns if np.issubdtype(df[c].dropna().dtype, np.number)]
    if not numeric_fields:
        print("No numeric fields found in the selected record set for EDA.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field '@id' for EDA: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        # Filter records above threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical/other field (first non-numeric field)
        group_fields = [c for c in df.columns if c not in numeric_fields]
        group_field = group_fields[0] if group_fields else None
        if group_field is not None:
            print(f"Grouping by field '@id': {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
            display(grouped.head())
        else:
            print("No group field found for grouping in this record set.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a simple histogram of the selected numeric field—and, if available, a box plot by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df_keys and numeric_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot (grouped if group_field is available)
    if group_field is not None:
        plt.figure(figsize=(10,6))
        top_groups = filtered_df[group_field].value_counts().index[:5]
        boxplot_df = filtered_df[filtered_df[group_field].isin(top_groups)]
        sns.boxplot(data=boxplot_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load a machine-actionable FAIR dataset defined by a Croissant schema using the `mlcroissant` library, and reference all entities (record sets, fields) using their `@id`. We explored metadata, auto-discovered record sets, loaded data into DataFrames, applied EDA, and visualized distributions.

**Summary of key steps:**
- Loading of metadata and inspection of dataset license, description, contents
- Examination of available record sets and their fields (all referenced by `@id`)
- Loading and EDA with dynamic field selection (threshold filtering, normalization, groupwise aggregation)
- Visualizations of numeric field distributions

This demonstrates the power of interoperable structured data and reproducible exploration pipelines enabled by the Croissant format and `mlcroissant` tools.